In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/label_mapping.json
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0079.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0018.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0032.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0049.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0215.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0064.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0016.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0114.jpg
/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup/public_0090.jpg

In [2]:
%%writefile feature_utils.py
"""
feature_utils.py
Các hàm dùng chung: segment lá, trích đặc trưng (color + texture + lesion),
và augmentation ảnh truyền thống (không dùng Deep Learning).
"""

import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern


# ============================================================
# 1. TIỀN XỬ LÝ: resize giữ tỉ lệ (pad) + segment lá bằng GrabCut
# ============================================================

def resize_with_pad(img_bgr, target=512):
    """Resize giữ nguyên tỉ lệ khung hình, pad viền đen cho đủ target x target.
    Tránh làm méo hình dạng vết bệnh (rất quan trọng vì aspect ratio ảnh gốc
    dao động 0.31 - 2.72 theo kết quả EDA)."""
    h, w = img_bgr.shape[:2]
    scale = target / max(h, w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img_bgr, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target, target, 3), dtype=np.uint8)
    top = (target - nh) // 2
    left = (target - nw) // 2
    canvas[top:top + nh, left:left + nw] = resized
    return canvas


def segment_leaf_grabcut(img_bgr, margin_ratio=0.06, iterations=3):
    """Tách vùng lá (foreground) khỏi nền (ruộng/nước/trời/tay) bằng GrabCut,
    khởi tạo bằng 1 rect ở giữa ảnh (giả định người chụp canh lá ở trung tâm).
    Trả về mask nhị phân (1 = lá, 0 = nền).

    Đây là kỹ thuật segmentation không giám sát, không hoàn hảo 100% với ảnh
    field phức tạp, nhưng loại bỏ được phần lớn nhiễu nền (bầu trời, nước,
    đất) mà color histogram/texture toàn ảnh sẽ bị nhiễu nếu không xử lý.
    """
    h, w = img_bgr.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    mx, my = int(w * margin_ratio), int(h * margin_ratio)
    rect = (mx, my, w - 2 * mx, h - 2 * my)

    try:
        cv2.grabCut(img_bgr, mask, rect, bgd_model, fgd_model,
                    iterations, cv2.GC_INIT_WITH_RECT)
        fg_mask = np.where((mask == 2) | (mask == 0), 0, 1).astype(np.uint8)
    except cv2.error:
        fg_mask = np.ones((h, w), dtype=np.uint8)

    coverage = fg_mask.sum() / (h * w)
    # GrabCut lỗi/thất bại (mask quá nhỏ hoặc quá lớn) -> fallback dùng cả ảnh
    if coverage < 0.05 or coverage > 0.97:
        fg_mask = np.ones((h, w), dtype=np.uint8)

    return fg_mask


# ============================================================
# 2. TRÍCH ĐẶC TRƯNG: color + texture (GLCM/LBP) + lesion shape
# ============================================================

def _safe_hist(values, bins, value_range):
    if values.size == 0:
        return np.zeros(bins)
    hist, _ = np.histogram(values, bins=bins, range=value_range, density=True)
    return hist


def extract_color_features(img_bgr, mask):
    """Đặc trưng màu: mean/std theo HSV + Lab, histogram H/S/V thô,
    chỉ tính trên pixel thuộc vùng lá (mask==1)."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

    idx = mask.astype(bool)
    if idx.sum() == 0:
        idx = np.ones_like(mask, dtype=bool)

    feats = {}
    for name, ch in zip(["H", "S", "V"], cv2.split(hsv)):
        vals = ch[idx].astype(np.float32)
        feats[f"{name}_mean"] = vals.mean()
        feats[f"{name}_std"] = vals.std()
    for name, ch in zip(["L", "a", "b"], cv2.split(lab)):
        vals = ch[idx].astype(np.float32)
        feats[f"{name}_mean"] = vals.mean()
        feats[f"{name}_std"] = vals.std()

    h_hist = _safe_hist(hsv[:, :, 0][idx], bins=8, value_range=(0, 180))
    s_hist = _safe_hist(hsv[:, :, 1][idx], bins=8, value_range=(0, 255))
    v_hist = _safe_hist(hsv[:, :, 2][idx], bins=8, value_range=(0, 255))
    for i, v in enumerate(h_hist):
        feats[f"hhist_{i}"] = v
    for i, v in enumerate(s_hist):
        feats[f"shist_{i}"] = v
    for i, v in enumerate(v_hist):
        feats[f"vhist_{i}"] = v

    return feats


def extract_texture_features(img_bgr, mask):
    """GLCM (contrast, homogeneity, energy, correlation, dissimilarity, ASM)
    + LBP histogram (uniform), tính trên vùng lá đã segment (nền được set về 0
    trước khi tính để giảm ảnh hưởng, GLCM/LBP vẫn nhạy 1 phần với biên mask
    nhưng đây là đánh đổi chấp nhận được để loại phần lớn nhiễu nền)."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_masked = gray.copy()
    gray_masked[mask == 0] = 0

    # Giảm số mức xám để GLCM tính nhanh + ổn định
    levels = 32
    gray_q = (gray_masked.astype(np.float32) / 256 * levels).astype(np.uint8)
    gray_q = np.clip(gray_q, 0, levels - 1)

    glcm = graycomatrix(gray_q, distances=[1, 3], angles=[0, np.pi / 2],
                         levels=levels, symmetric=True, normed=True)

    feats = {}
    for prop in ["contrast", "homogeneity", "energy", "correlation",
                 "dissimilarity", "ASM"]:
        vals = graycoprops(glcm, prop)
        feats[f"glcm_{prop}"] = vals.mean()

    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_vals = lbp[mask.astype(bool)] if mask.sum() > 0 else lbp.ravel()
    n_bins = 10  # uniform LBP với P=8 -> 10 patterns
    lbp_hist, _ = np.histogram(lbp_vals, bins=n_bins, range=(0, 10), density=True)
    for i, v in enumerate(lbp_hist):
        feats[f"lbp_{i}"] = v

    return feats


def extract_lesion_features(img_bgr, mask):
    """Tách vùng 'bệnh' (không xanh: vàng/nâu/trắng) khỏi vùng lá khỏe (xanh)
    trong phạm vi mask lá, rồi đo diện tích, số vùng liên thông, độ kéo dài.
    Đây là nhóm đặc trưng quan trọng để phân biệt các bệnh có vệt/đốm khác
    hình dạng (Leaf Blast: đốm thoi, Narrow Brown Leaf Spot: vệt mảnh dài,
    Leaf scald: dải zigzag...)."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    leaf_area = max(mask.sum(), 1)

    # Vùng xanh khỏe: Hue ~ 35-90 (OpenCV H: 0-180), S đủ cao
    h, s, v = cv2.split(hsv)
    healthy_green = ((h >= 30) & (h <= 95) & (s >= 60)).astype(np.uint8)
    lesion = ((mask == 1) & (healthy_green == 0)).astype(np.uint8)

    lesion_ratio = float(lesion.sum()) / leaf_area

    feats = {"lesion_ratio": lesion_ratio}

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        lesion, connectivity=8)
    # bỏ background label 0
    areas = stats[1:, cv2.CC_STAT_AREA] if num_labels > 1 else np.array([])
    areas = areas[areas >= 5]  # loại nhiễu quá nhỏ

    feats["lesion_num_components"] = len(areas)
    feats["lesion_area_mean"] = float(areas.mean()) if len(areas) else 0.0
    feats["lesion_area_std"] = float(areas.std()) if len(areas) else 0.0
    feats["lesion_area_max"] = float(areas.max()) if len(areas) else 0.0

    # Độ kéo dài (elongation) của vùng lesion lớn nhất -> phân biệt vệt dài mảnh
    # vs đốm tròn/thoi
    elongation = 0.0
    if len(areas):
        largest_label = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        comp_mask = (labels == largest_label).astype(np.uint8) * 255
        contours, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        if contours and len(contours[0]) >= 5:
            (_, _), (major, minor), _ = cv2.fitEllipse(contours[0])
            elongation = major / max(minor, 1e-3)
    feats["lesion_elongation"] = elongation

    # Màu trung bình của vùng lesion (giúp phân biệt bệnh vàng/nâu/trắng)
    idx = lesion.astype(bool)
    if idx.sum() > 0:
        lesion_hsv = hsv[idx]
        feats["lesion_H_mean"] = float(lesion_hsv[:, 0].mean())
        feats["lesion_S_mean"] = float(lesion_hsv[:, 1].mean())
        feats["lesion_V_mean"] = float(lesion_hsv[:, 2].mean())
    else:
        feats["lesion_H_mean"] = 0.0
        feats["lesion_S_mean"] = 0.0
        feats["lesion_V_mean"] = 0.0

    return feats


def extract_all_features(img_bgr, target_size=512):
    """Pipeline đầy đủ cho 1 ảnh: resize+pad -> segment -> color+texture+lesion."""
    img = resize_with_pad(img_bgr, target=target_size)
    mask = segment_leaf_grabcut(img)

    feats = {}
    feats["leaf_coverage"] = float(mask.sum()) / mask.size
    feats.update(extract_color_features(img, mask))
    feats.update(extract_texture_features(img, mask))
    feats.update(extract_lesion_features(img, mask))
    return feats


# ============================================================
# 3. AUGMENTATION TRUYỀN THỐNG (cho 2 class thiếu dữ liệu)
# ============================================================

def augment_image(img_bgr, seed=None):
    """1 phép augment ngẫu nhiên: flip + rotate nhẹ + color jitter nhẹ.
    Dùng cho các class thiếu dữ liệu (Narrow Brown Leaf Spot, Healthy Rice Leaf)
    để tăng số ảnh THẬT (không phải synthetic) trước khi trích đặc trưng."""
    rng = np.random.RandomState(seed)
    img = img_bgr.copy()

    if rng.rand() < 0.5:
        img = cv2.flip(img, 1)  # lật ngang
    if rng.rand() < 0.3:
        img = cv2.flip(img, 0)  # lật dọc

    angle = rng.uniform(-20, 20)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    # jitter độ sáng/tương phản nhẹ (không đổi hue để giữ đặc trưng bệnh)
    alpha = rng.uniform(0.9, 1.1)   # contrast
    beta = rng.uniform(-15, 15)     # brightness
    img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    return img


Writing feature_utils.py


In [3]:
%%writefile 01_extract_features.py

"""
01_extract_features.py

Chạy trên Kaggle (hoặc local sau khi tải dataset về).
Output:
    - train_features.csv   (đặc trưng + label, đã augment 2 class thiếu)
    - test_features.csv    (đặc trưng, không có label)

Cách chạy:
    python 01_extract_features.py

Sửa các biến trong phần CONFIG bên dưới cho đúng đường dẫn dataset của bạn.
"""

import os
import json
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from feature_utils import extract_all_features, augment_image

# ============ CONFIG ============
DATA_DIR = "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup"     # <-- SỬA đường dẫn thật
TRAIN_DIR = os.path.join(DATA_DIR, "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/train")
TEST_DIR = os.path.join(DATA_DIR, "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/test_warmup")  # <-- kiểm tra đúng tên thư mục test
LABEL_MAP_PATH = os.path.join(DATA_DIR, "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/label_mapping.json")

OUT_DIR = "./features_output"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_SIZE = 256      # resize ảnh về TARGET_SIZE x TARGET_SIZE trước khi trích đặc trưng
                        # (256 để chạy nhanh hơn, có thể tăng lên 384/512 nếu không giới hạn thời gian)
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Số ảnh augment thêm cho các class thiếu dữ liệu sẽ được tính tự động
# dựa trên AUGMENT_TARGET_PERCENTILE của phân bố class (mặc định: median)
AUGMENT_TARGET_PERCENTILE = 50


def load_image(path):
    img = cv2.imread(path)
    if img is None:
        # fallback cho vài trường hợp OpenCV không đọc được (unicode path, v.v.)
        from PIL import Image
        pil_img = Image.open(path).convert("RGB")
        img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    return img


def process_train():
    with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
        label_mapping = json.load(f)

    classes = sorted(os.listdir(TRAIN_DIR))
    class_files = {}
    for cls in classes:
        cls_dir = os.path.join(TRAIN_DIR, cls)
        if not os.path.isdir(cls_dir):
            continue
        files = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir)
                  if Path(f).suffix.lower() in VALID_EXT]
        class_files[cls] = files

    counts = {c: len(f) for c, f in class_files.items()}
    target_count = int(np.percentile(list(counts.values()), AUGMENT_TARGET_PERCENTILE))
    print("Số lượng ảnh mỗi class:", counts)
    print(f"Target augment (percentile {AUGMENT_TARGET_PERCENTILE}): {target_count}")

    rows = []
    for cls, files in class_files.items():
        label = label_mapping[cls]
        n_have = len(files)
        n_extra = max(0, target_count - n_have)

        print(f"\nClass '{cls}': {n_have} ảnh gốc, augment thêm {n_extra} ảnh")

        # --- ảnh gốc ---
        for p in tqdm(files, desc=f"[{cls}] gốc"):
            img = load_image(p)
            feats = extract_all_features(img, target_size=TARGET_SIZE)
            feats.update({"image_path": p, "class": cls, "label": label,
                          "is_augmented": 0, "source_image": p})
            rows.append(feats)

        # --- ảnh augment thêm (chỉ áp dụng khi class thiếu dữ liệu) ---
        if n_extra > 0:
            for i in tqdm(range(n_extra), desc=f"[{cls}] augment"):
                src_path = files[i % n_have]
                img = load_image(src_path)
                aug_img = augment_image(img, seed=i)
                feats = extract_all_features(aug_img, target_size=TARGET_SIZE)
                feats.update({"image_path": f"{src_path}#aug{i}", "class": cls,
                              "label": label, "is_augmented": 1,
                              "source_image": src_path})
                rows.append(feats)

    df = pd.DataFrame(rows)
    out_path = os.path.join(OUT_DIR, "train_features.csv")
    df.to_csv(out_path, index=False)
    print(f"\n✅ Đã lưu {out_path} — {len(df)} dòng, {df['class'].nunique()} class")
    print(df.groupby("class")["is_augmented"].agg(["count", "sum"]))


def process_test():
    if not os.path.isdir(TEST_DIR):
        print(f"⚠️ Không tìm thấy thư mục test tại {TEST_DIR}, bỏ qua bước này.")
        return

    files = [f for f in os.listdir(TEST_DIR) if Path(f).suffix.lower() in VALID_EXT]
    rows = []
    for f in tqdm(files, desc="test_images"):
        path = os.path.join(TEST_DIR, f)
        img = load_image(path)
        feats = extract_all_features(img, target_size=TARGET_SIZE)
        feats.update({"image_id": f})
        rows.append(feats)

    df = pd.DataFrame(rows)
    out_path = os.path.join(OUT_DIR, "test_features.csv")
    df.to_csv(out_path, index=False)
    print(f"\n✅ Đã lưu {out_path} — {len(df)} dòng")


if __name__ == "__main__":
    print("=== TRÍCH ĐẶC TRƯNG TẬP TRAIN ===")
    process_train()
    print("\n=== TRÍCH ĐẶC TRƯNG TẬP TEST ===")
    process_test()



Writing 01_extract_features.py


In [13]:
%%writefile 03_dedup_check.py
"""
03_dedup_check.py

Kiểm tra ảnh GẦN TRÙNG LẶP (near-duplicate) trong tập train — ví dụ nhiều ảnh
là CÙNG 1 chiếc lá chụp liên tiếp (burst mode) hoặc crop từ cùng 1 ảnh gốc.
Đây là nguồn rò rỉ CV rất phổ biến với dataset ảnh thực vật, khác hẳn với
leakage do augmentation (đã fix ở bước trước).

Cách hoạt động: tính perceptual hash (phash) cho từng ảnh gốc trong train/,
gộp các ảnh có khoảng cách Hamming <= DUPLICATE_THRESHOLD vào cùng 1 cụm
(union-find). Nếu số cụm lớn xuất hiện nhiều, gần như chắc chắn đây là lý do
CV cao hơn nhiều so với leaderboard thật.

Output: features_output/duplicate_clusters.csv (image_path -> cluster_id)
File này sẽ được 02_train_predict.py tự động dùng làm group CV "thật" nếu tồn tại.

Cần cài: pip install imagehash --break-system-packages
Chạy: python 03_dedup_check.py
"""

import os
from pathlib import Path
from collections import defaultdict

import pandas as pd
from PIL import Image
import imagehash

# ============ CONFIG ============
DATA_DIR = "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup"      # <-- SỬA đường dẫn thật
TRAIN_DIR = os.path.join(DATA_DIR, "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/train")
OUT_DIR = "./features_output"
os.makedirs(OUT_DIR, exist_ok=True)

HASH_SIZE = 16            # độ chi tiết hash (16 -> hash 256-bit, khá nhạy)
DUPLICATE_THRESHOLD = 6   # khoảng cách Hamming tối đa để coi là "gần trùng"
                          # tăng lên nếu muốn bắt cả ảnh crop/resize khác nhau nhiều hơn
                          # giảm xuống nếu thấy false positive (ảnh khác nhau bị gộp nhầm)

VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def compute_hashes():
    records = []
    classes = sorted(os.listdir(TRAIN_DIR))
    for cls in classes:
        cls_dir = os.path.join(TRAIN_DIR, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in os.listdir(cls_dir):
            if Path(f).suffix.lower() not in VALID_EXT:
                continue
            path = os.path.join(cls_dir, f)
            try:
                h = imagehash.phash(Image.open(path), hash_size=HASH_SIZE)
            except Exception as e:
                print("Lỗi đọc", path, e)
                continue
            records.append({"path": path, "class": cls, "hash": h})
    return records


def cluster_duplicates(records):
    n = len(records)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    # So sánh trong cùng class trước (duplicate thường nằm cùng class, và
    # nhanh hơn nhiều so với so sánh toàn bộ N^2 ảnh)
    by_class = defaultdict(list)
    for i, r in enumerate(records):
        by_class[r["class"]].append(i)

    for cls, idxs in by_class.items():
        for a in range(len(idxs)):
            for b in range(a + 1, len(idxs)):
                i, j = idxs[a], idxs[b]
                dist = records[i]["hash"] - records[j]["hash"]
                if dist <= DUPLICATE_THRESHOLD:
                    union(i, j)

    clusters = defaultdict(list)
    for i in range(n):
        clusters[find(i)].append(i)
    return clusters


def main():
    print("Đang tính perceptual hash cho toàn bộ ảnh train (có thể mất vài phút)...")
    records = compute_hashes()
    print(f"Tổng số ảnh: {len(records)}")

    clusters = cluster_duplicates(records)
    dup_clusters = {k: v for k, v in clusters.items() if len(v) > 1}

    total_dup_images = sum(len(v) for v in dup_clusters.values())
    print(f"\nSố cụm ảnh gần trùng (>=2 ảnh): {len(dup_clusters)}")
    print(f"Tổng số ảnh nằm trong 1 cụm trùng: {total_dup_images} / {len(records)} "
          f"({100 * total_dup_images / len(records):.1f}%)")

    rows = []
    for cid, idxs in clusters.items():
        for i in idxs:
            rows.append({"image_path": records[i]["path"],
                         "class": records[i]["class"],
                         "cluster_id": cid,
                         "cluster_size": len(idxs)})
    df = pd.DataFrame(rows)
    out_path = os.path.join(OUT_DIR, "duplicate_clusters.csv")
    df.to_csv(out_path, index=False)
    print(f"\n✅ Đã lưu {out_path}")

    if dup_clusters:
        print("\nTop 15 cụm lớn nhất (kiểm tra xem có phải cùng 1 lá chụp nhiều lần không):")
        top = (df.sort_values("cluster_size", ascending=False)
                 .drop_duplicates("cluster_id")[["cluster_id", "class", "cluster_size"]]
                 .head(15))
        print(top.to_string(index=False))

        # Cảnh báo nếu 1 cụm chứa ảnh thuộc NHIỀU class khác nhau -> đáng ngờ hơn
        cross_class = df.groupby("cluster_id")["class"].nunique()
        n_cross = (cross_class > 1).sum()
        if n_cross > 0:
            print(f"\n⚠️ Có {n_cross} cụm chứa ảnh thuộc >1 class khác nhau — "
                  f"đáng xem lại thủ công, có thể là ảnh bị gán nhầm nhãn hoặc "
                  f"threshold đang quá lỏng.")
    else:
        print("\n✅ Không phát hiện ảnh gần trùng lặp đáng kể trong tập train.")


if __name__ == "__main__":
    main()


Overwriting 03_dedup_check.py


In [14]:
%%writefile 02_train_predict.py

"""
02_train_predict.py

Đọc train_features.csv / test_features.csv (từ bước 01), xử lý mất cân bằng
(SMOTE + class_weight), Stratified GROUP K-Fold CV tối ưu trực tiếp Macro-F1
(nhóm theo ảnh gốc để tránh rò rỉ dữ liệu giữa ảnh gốc và bản augment của nó),
so sánh Random Forest vs XGBoost, chọn model tốt nhất, train lại trên toàn bộ
train rồi dự đoán tập test theo đúng format submission.

Cách chạy:
    python 02_train_predict.py
"""

import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# ============ CONFIG ============
FEATURES_DIR = "./features_output"
TRAIN_CSV = os.path.join(FEATURES_DIR, "train_features.csv")
TEST_CSV = os.path.join(FEATURES_DIR, "test_features.csv")
LABEL_MAP_PATH = "/kaggle/input/competitions/fptu-can-tho-olympic-ai-2026-vong-lo-i-warmup/label_mapping.json"  # <-- SỬA đường dẫn thật

OUT_DIR = "./train_output"
os.makedirs(OUT_DIR, exist_ok=True)

N_FOLDS = 5
RANDOM_STATE = 42

NON_FEATURE_COLS = {"image_path", "class", "label", "is_augmented", "image_id",
                     "source_image"}


DUPLICATE_CLUSTERS_CSV = os.path.join(FEATURES_DIR, "duplicate_clusters.csv")


def load_data():
    df_train = pd.read_csv(TRAIN_CSV)
    df_test = pd.read_csv(TEST_CSV)
    feature_cols = [c for c in df_train.columns if c not in NON_FEATURE_COLS]
    X = df_train[feature_cols].values
    y = df_train["label"].values

    # Group CV: mặc định group theo source_image (ảnh gốc + bản augment của nó).
    # Nếu đã chạy 03_dedup_check.py, dùng cluster_id (ảnh gần trùng lặp thật
    # trong dataset gốc) làm group -> group "thật" hơn nhiều, tránh leakage do
    # nhiều ảnh khác tên file nhưng là cùng 1 lá chụp liên tiếp.
    if os.path.exists(DUPLICATE_CLUSTERS_CSV):
        dup_df = pd.read_csv(DUPLICATE_CLUSTERS_CSV)
        cluster_map = dict(zip(dup_df["image_path"], dup_df["cluster_id"]))
        groups = df_train["source_image"].map(cluster_map)
        # ảnh không có trong cluster_map (không nên xảy ra, nhưng fallback an toàn)
        missing = groups.isna()
        groups[missing] = df_train.loc[missing, "source_image"]
        groups = groups.values
        print(f"✅ Dùng duplicate_clusters.csv làm group CV "
              f"({dup_df['cluster_id'].nunique()} cụm, "
              f"{(dup_df.groupby('cluster_id').size() > 1).sum()} cụm có >1 ảnh)")
    else:
        groups = df_train["source_image"].values
        print("⚠️ Không tìm thấy duplicate_clusters.csv — group CV theo source_image "
              "(chỉ tránh leak augmentation, CHƯA kiểm tra ảnh gần trùng lặp gốc). "
              "Khuyến khích chạy 03_dedup_check.py trước để có đánh giá CV tin cậy hơn.")

    is_augmented = df_train["is_augmented"].values
    X_test = df_test[feature_cols].values
    return X, y, groups, is_augmented, X_test, df_test, feature_cols


def _fit_smote(X_tr, y_tr):
    min_class_count = pd.Series(y_tr).value_counts().min()
    k_neighbors = max(1, min(5, min_class_count - 1))
    if min_class_count > 1:
        sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)
        return sm.fit_resample(X_tr, y_tr)
    return X_tr, y_tr


def make_rf(y_tr_bal):
    return RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)


sample_weight_fn_holder = {}  # xgboost không có class_weight, dùng sample_weight


def make_xgb(y_tr_bal):
    classes = np.unique(y_tr_bal)
    cw = compute_class_weight("balanced", classes=classes, y=y_tr_bal)
    weight_map = dict(zip(classes, cw))
    sample_weight_fn_holder["weights"] = np.array([weight_map[c] for c in y_tr_bal])
    return XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1)


def evaluate_grouped(X, y, groups, is_augmented, model_name, n_folds=N_FOLDS,
                      verbose=True):
    """Stratified GROUP K-Fold: ảnh gốc + bản augment của nó luôn cùng 1 fold.
    Khi CHẤM ĐIỂM validation, chỉ dùng ảnh GỐC (is_augmented==0) -> điểm CV
    phản ánh đúng khả năng dự đoán ảnh hoàn toàn mới, giống tập test thật.
    Train fold vẫn dùng cả gốc + augment + SMOTE để tận dụng augmentation."""
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True,
                                 random_state=RANDOM_STATE)
    scores = []
    all_val_true, all_val_pred = [], []

    for fold, (tr_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
        # loại augment ra khỏi validation, chỉ giữ ảnh gốc để chấm điểm
        val_idx_orig = val_idx[is_augmented[val_idx] == 0]
        if len(val_idx_orig) == 0:
            continue

        X_tr, X_val = X[tr_idx], X[val_idx_orig]
        y_tr, y_val = y[tr_idx], y[val_idx_orig]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_val_s = scaler.transform(X_val)

        X_tr_bal, y_tr_bal = _fit_smote(X_tr_s, y_tr)

        if model_name == "rf":
            model = make_rf(y_tr_bal)
            model.fit(X_tr_bal, y_tr_bal)
        else:
            model = make_xgb(y_tr_bal)
            model.fit(X_tr_bal, y_tr_bal, sample_weight=sample_weight_fn_holder["weights"])

        pred = model.predict(X_val_s)
        f1 = f1_score(y_val, pred, average="macro")
        scores.append(f1)
        all_val_true.extend(y_val.tolist())
        all_val_pred.extend(pred.tolist())
        if verbose:
            print(f"  Fold {fold+1}: Macro-F1 = {f1:.4f} (n_val_goc={len(val_idx_orig)})")

    return np.mean(scores), np.std(scores), np.array(all_val_true), np.array(all_val_pred)


def train_final_and_predict(X, y, X_test, model_name):
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    X_test_s = scaler.transform(X_test)

    X_bal, y_bal = _fit_smote(X_s, y)

    if model_name == "rf":
        model = make_rf(y_bal)
        model.fit(X_bal, y_bal)
    else:
        model = make_xgb(y_bal)
        model.fit(X_bal, y_bal, sample_weight=sample_weight_fn_holder["weights"])

    test_pred = model.predict(X_test_s)
    return model, test_pred


def main():
    X, y, groups, is_augmented, X_test, df_test, feature_cols = load_data()
    print(f"Train: {X.shape}, Test: {X_test.shape}, số class: {len(np.unique(y))}")
    print(f"Số ảnh gốc: {(is_augmented == 0).sum()}, số ảnh augment: {(is_augmented == 1).sum()}")

    print("\n=== Đánh giá Random Forest (StratifiedGroupKFold + SMOTE + class_weight) ===")
    rf_mean, rf_std, rf_true, rf_pred = evaluate_grouped(X, y, groups, is_augmented, "rf")
    print(f"RF Macro-F1 (chỉ tính trên ảnh gốc): {rf_mean:.4f} +- {rf_std:.4f}")

    print("\n=== Đánh giá XGBoost (StratifiedGroupKFold + SMOTE + sample_weight) ===")
    xgb_mean, xgb_std, xgb_true, xgb_pred = evaluate_grouped(X, y, groups, is_augmented, "xgb")
    print(f"XGB Macro-F1 (chỉ tính trên ảnh gốc): {xgb_mean:.4f} +- {xgb_std:.4f}")

    if rf_mean >= xgb_mean:
        best_model_name, best_true, best_pred = "rf", rf_true, rf_pred
    else:
        best_model_name, best_true, best_pred = "xgb", xgb_true, xgb_pred
    print(f"\n>>> Model tốt hơn: {best_model_name.upper()}")

    with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
        label_mapping = json.load(f)
    inv_label_mapping = {v: k for k, v in label_mapping.items()}
    target_names = [inv_label_mapping[i] for i in sorted(inv_label_mapping)]

    print("\n=== Classification report (gộp toàn bộ fold, chỉ ảnh gốc) ===")
    print(classification_report(best_true, best_pred, target_names=target_names, zero_division=0))
    print("=== Confusion matrix ===")
    cm = confusion_matrix(best_true, best_pred)
    print(pd.DataFrame(cm, index=target_names, columns=target_names))

    print("\n=== Train model cuối cùng trên toàn bộ train, dự đoán test ===")
    model, test_pred = train_final_and_predict(X, y, X_test, best_model_name)

    submission = pd.DataFrame({
        "image_id": df_test["image_id"],
        "label": test_pred
    })
    out_path = os.path.join(OUT_DIR, "submission.csv")
    submission.to_csv(out_path, index=False)
    print(f"\n✅ Đã lưu {out_path}")
    print(submission.head())
    print("\nPhân bố nhãn dự đoán trên test:")
    print(submission["label"].map(inv_label_mapping).value_counts())


if __name__ == "__main__":
    main()

Overwriting 02_train_predict.py


In [5]:
!python 01_extract_features.py

=== TRÍCH ĐẶC TRƯNG TẬP TRAIN ===
Số lượng ảnh mỗi class: {'Bacterial Leaf Blight': 226, 'Brown Spot': 289, 'Healthy Rice Leaf': 193, 'Leaf Blast': 360, 'Leaf scald': 220, 'Narrow Brown Leaf Spot': 126, 'Rice Hispa': 260, 'Sheath Blight': 352}
Target augment (percentile 50): 243

Class 'Bacterial Leaf Blight': 226 ảnh gốc, augment thêm 17 ảnh
[Bacterial Leaf Blight] augment: 100%|██████████| 17/17 [00:11<00:00,  1.51it/s]

Class 'Brown Spot': 289 ảnh gốc, augment thêm 0 ảnh
[Brown Spot] gốc: 100%|███████████████████████| 289/289 [02:54<00:00,  1.65it/s]

Class 'Healthy Rice Leaf': 193 ảnh gốc, augment thêm 50 ảnh
[Healthy Rice Leaf] augment: 100%|██████████████| 50/50 [00:32<00:00,  1.56it/s]

Class 'Leaf Blast': 360 ảnh gốc, augment thêm 0 ảnh
[Leaf Blast] gốc: 100%|███████████████████████| 360/360 [04:07<00:00,  1.46it/s]

Class 'Leaf scald': 220 ảnh gốc, augment thêm 23 ảnh
[Leaf scald] augment: 100%|█████████████████████| 23/23 [00:14<00:00,  1.57it/s]

Class 'Narrow Brown Leaf Spo

In [15]:
!python 03_dedup_check.py

Đang tính perceptual hash cho toàn bộ ảnh train (có thể mất vài phút)...
Tổng số ảnh: 2026

Số cụm ảnh gần trùng (>=2 ảnh): 199
Tổng số ảnh nằm trong 1 cụm trùng: 424 / 2026 (20.9%)

✅ Đã lưu ./features_output/duplicate_clusters.csv

Top 15 cụm lớn nhất (kiểm tra xem có phải cùng 1 lá chụp nhiều lần không):
 cluster_id                  class  cluster_size
       2006          Sheath Blight             4
        155  Bacterial Leaf Blight             4
        203  Bacterial Leaf Blight             3
       1946          Sheath Blight             3
       1667             Rice Hispa             3
       1612             Rice Hispa             3
        938             Leaf Blast             3
        624      Healthy Rice Leaf             3
       1617             Rice Hispa             3
       1210             Leaf scald             3
       1477             Rice Hispa             3
        434             Brown Spot             3
       1389 Narrow Brown Leaf Spot             3
     

In [6]:
!python 02_train_predict.py

Train: (2233, 62), Test: (300, 62), số class: 8
Số ảnh gốc: 2026, số ảnh augment: 207

=== Đánh giá Random Forest (StratifiedGroupKFold + SMOTE + class_weight) ===
  Fold 1: Macro-F1 = 0.8685 (n_val_goc=405)
  Fold 2: Macro-F1 = 0.9083 (n_val_goc=405)
  Fold 3: Macro-F1 = 0.9074 (n_val_goc=405)
  Fold 4: Macro-F1 = 0.9069 (n_val_goc=405)
  Fold 5: Macro-F1 = 0.9037 (n_val_goc=406)
RF Macro-F1 (chỉ tính trên ảnh gốc): 0.8990 +- 0.0153

=== Đánh giá XGBoost (StratifiedGroupKFold + SMOTE + sample_weight) ===
  Fold 1: Macro-F1 = 0.8962 (n_val_goc=405)
  Fold 2: Macro-F1 = 0.9112 (n_val_goc=405)
  Fold 3: Macro-F1 = 0.8893 (n_val_goc=405)
  Fold 4: Macro-F1 = 0.9252 (n_val_goc=405)
  Fold 5: Macro-F1 = 0.9149 (n_val_goc=406)
XGB Macro-F1 (chỉ tính trên ảnh gốc): 0.9074 +- 0.0130

>>> Model tốt hơn: XGB

=== Classification report (gộp toàn bộ fold, chỉ ảnh gốc) ===
                        precision    recall  f1-score   support

 Bacterial Leaf Blight       0.89      0.94      0.91       22